# 2b - AI wrote it. Is the number right?

**Time:** 35 minutes  |  **AI: OFF** (no Gemini, no ChatGPT in this notebook)

**What you will do**
- Part 1 (10 min): test the 3-fund risk formula with a known case and a crisis.
- Part 2 (25 min): check **real AI code** on **real data** from a terminal export, and find why the number is wrong.

No new Python to write from scratch. The job here is to decide if a number is right. Each step has a **Stuck?** hint: read it only after you have tried.

## Part 1 - Test the formula you will trust

The function below is the 3-fund formula from 2a, in one place. Before you trust it on a client, test it on cases where you know the answer without code.

In [ ]:
import numpy as np

w = np.array([0.20, 0.30, 0.50])      # client weights in A, B, C
vol = np.array([0.15, 0.20, 0.25])    # volatilities
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])


def portfolio_vol(weights, vol, corr):
    cov = np.diag(vol) @ corr @ np.diag(vol)   # every pair: vol x correlation x vol
    return np.sqrt(weights @ cov @ weights)    # variance -> volatility


print(f"Client 20 / 30 / 50: {portfolio_vol(w, vol, corr):.2%}")
print(f"Weighted average of the vols: {np.sum(w * vol):.2%}")

You should get **18.44%** against **21.50%** for the weighted average: diversification saves about 3 points.

### Known case: all correlations 1

A **known case** is a situation where you know the answer without code. If all correlations are 1, the funds move in lockstep and nothing cancels: the portfolio vol must be the weighted average, **21.5%** exactly. If the code gives anything else, the code is wrong. (Checks card, entry 5.)

Make a 3 x 3 array full of ones with `np.ones((3, 3))` and run the function on it.

In [ ]:
corr_one = ...   # YOUR CODE
print(f"All correlations 1: {portfolio_vol(w, vol, corr_one):.2%}")

### Crisis: all correlations 0.9

In a crisis, everything falls together. `np.full((3, 3), 0.9)` makes a 3 x 3 array of 0.9; `np.fill_diagonal(crisis, 1)` puts 1 on the diagonal (a fund always moves with itself).

You should get **20.88%**: almost back to 21.5%. Diversification disappears when you need it most.

**Stuck?** Two lines: `crisis = np.full(...)`, then `np.fill_diagonal(crisis, 1)` on its own line.

In [ ]:
crisis = ...   # YOUR CODE
...            # YOUR CODE
print(f"Crisis: {portfolio_vol(w, vol, crisis):.2%}")

## Part 2 - Real AI code, real data

A colleague needs the risk of each of the 20 stocks and of the equal-weight portfolio. He asked an AI assistant, word for word:

> *I have a CSV file stocks_20.csv with daily prices of 20 stocks (first column Date, then one column per ticker). Write Python code with pandas and numpy that computes the volatility of each stock and the volatility of a portfolio with equal weights. Only the code, no explanation.*

We sent this prompt (and two other wordings) six times, to two AI models, in November 2026. **The code was correct every time.** The cell below is one of those answers, **unchanged**, except the file name.

The data desk sends the prices exported from the market terminal: `stocks_20_raw.csv`. Run the AI's code on it.

In [ ]:
import numpy as np
import pandas as pd

TRADING_DAYS = 252

prices = (
    pd.read_csv("https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/stocks_20_raw.csv", parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
    .astype(float)
)

returns = np.log(prices / prices.shift(1)).dropna()

# Annualized volatility of each stock
stock_vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)

# Equal-weight portfolio volatility
n_assets = returns.shape[1]
weights = np.full(n_assets, 1.0 / n_assets)
cov_annual = returns.cov(ddof=1).to_numpy() * TRADING_DAYS
portfolio_vol = np.sqrt(weights @ cov_annual @ weights)

print(stock_vol.sort_values(ascending=False).to_string())
print(f"Equal-weight portfolio volatility: {portfolio_vol:.4f}")

**Your prediction:** the portfolio volatility is about **19.2%**. In the opener of 2a we found 15.7% for the same 20 stocks. Is 19.2% wrong? Would you have noticed it on its own?

### Step 1 - Range check on each stock

A **range check**: large US stocks usually have a volatility between about 15% and 50% a year. Look at the list printed above (sorted from the highest). Which stocks are far outside that range?

Write the names and their volatility here: ...

**Stuck?** Look at the top of the list. Would Alphabet (Google) really move more than 100% a year?

### Step 2 - Look at the extremes

When a number is too big, look at the most extreme day behind it. The cell below finds the worst day of each suspect stock: its date and its return. Run it.

In [ ]:
import pandas as pd

raw = pd.read_csv("https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/stocks_20_raw.csv", index_col="Date")
worst_day = raw.pct_change().idxmin()       # the date of each stock's worst day
worst_return = raw.pct_change().min()       # the return on that day

for name in ["GOOGL", "AMZN", "NVDA", "WMT"]:
    print(f"{name}: {worst_return[name]:.1%} on {worst_day[name]}")

**Your answer:** did Amazon really lose 95% in one day on 6 June 2022? Search the web for "Amazon June 2022" (this is allowed: it is not AI writing your answer). What happened on those four dates?

Write here: ...

### Step 3 - See it

The prices of Amazon around the split. A drop from about 2,447 to 125 overnight.

In [ ]:
print(raw.loc["2022-06-01":"2022-06-08", "AMZN"])

### Step 4 - Fix the data, not the code

The code was never wrong. The data was. Market data providers give two prices: the **raw** price, and the **adjusted** price, corrected for splits and dividends. Risk and returns must use adjusted prices. `stocks_20.csv` holds the adjusted prices.

In the cell below, the AI's code is copied again. Change only the file name: `stocks_20_raw.csv` becomes `stocks_20.csv`. Run it.

You should get **15.65%**, and every stock between 17% and 52%. (The opener of 2a gave 15.7%: the AI used log returns, a slightly different but accepted way. Two methods, close answers.)

In [ ]:
# Change ONLY the file name below: stocks_20_raw.csv -> stocks_20.csv
import numpy as np
import pandas as pd

TRADING_DAYS = 252

prices = (
    pd.read_csv("https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/stocks_20_raw.csv", parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
    .astype(float)
)

returns = np.log(prices / prices.shift(1)).dropna()

# Annualized volatility of each stock
stock_vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)

# Equal-weight portfolio volatility
n_assets = returns.shape[1]
weights = np.full(n_assets, 1.0 / n_assets)
cov_annual = returns.cov(ddof=1).to_numpy() * TRADING_DAYS
portfolio_vol = np.sqrt(weights @ cov_annual @ weights)

print(stock_vol.sort_values(ascending=False).to_string())
print(f"Equal-weight portfolio volatility: {portfolio_vol:.4f}")

### What happened

| | Raw export | Adjusted prices |
|---|---|---|
| Portfolio volatility | 19.2% | **15.65%** |
| Alphabet volatility | 139% | 33% |
| Did the code warn you? | No | - |

AI wrote correct code in seconds. It did not look at the data, and it could not know where your file came from. **Checking the number is your job**: a range check on every line, not only the total, and a look at the extremes.

## Go further (if you finish early)

### Go further 1 - Fix the raw file yourself

Undo the four splits in `raw`: every price **before** the split date must be divided by the split ratio (AMZN 20 on 2022-06-06, GOOGL 20 on 2022-07-18, NVDA 10 on 2024-06-10, WMT 3 on 2024-02-26). Hint: `raw.loc[raw.index < "2022-06-06", "AMZN"] /= 20`. Then check that the worst days are now real market days.

In [ ]:
# YOUR CODE

### Go further 2 - Why only 19.2%?

Four stocks show 115% to 139% volatility, yet the portfolio only goes from 15.65% to 19.2%. Explain in 3 lines. (Think about the weight of each stock, and whether the fake drops happen on the same day.)

Write here: ...

## What you learned

**Python**
- **Known case** with `np.ones((3, 3))`, **crisis** with `np.full` and `np.fill_diagonal`: test a function before you trust it.
- Reading real AI code: `read_csv`, returns, `std`, `cov`, `@`. You did not write it; you checked it.
- `pct_change().min()` and `.idxmin()` find the most extreme day, and its date.

**Finance**
- Known case: all correlations 1 gives the weighted average, 21.5%. Crisis: 20.88%. Diversification disappears when you need it most.
- **Raw vs adjusted prices.** A stock split divides the price, not the value. Raw prices turn a split into a fake 95% loss. Always use adjusted prices for returns and risk.
- The total looked fine (19.2%); the detail was absurd (139%). **Check every line, not only the total.**

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right